# Tutorial 80 — Retrieve by lexical similarity

[Tutorial 70](./70_semantic_retrieval.ipynb) searched by meaning. This tutorial
searches by wording using a TF-IDF lexical index, without loading an embedding
model.

Lexical retrieval is especially useful for exact identifiers, rare terms, and
short phrases. You will inspect the generated unigram and bigram vocabulary,
build a reusable TF-IDF representation, and rank chunks by lexical cosine
similarity.

[Tutorial 90](./90_code_aware_retrieval.ipynb) adds a third signal that follows
explicit relationships in source code rather than meaning or wording.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Prepare repository chunks

The following cells repeat the source-processing pipeline from
[Tutorial 40](./40_repository_files.ipynb) and
[Tutorial 50](./50_source_structure.ipynb) so this notebook remains
independently runnable. Lexical retrieval does not need an embedding tokenizer,
so the example uses a simple word counter only to keep very large source units
bounded. That counter is a teaching convenience, not QAChat's general
tokenization policy.


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)

print(f"Selected {len(report['selected_files'])} files:")

for i, file in enumerate(report["selected_files"]):
    print(f"{i}. {file}")


In [ ]:
source_files = qachat.read_repository_files(report)
units = qachat.extract_source_units(source_files)

print(f"Read {len(source_files)} files")
print(f"Natural units: {len(units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Build lexical terms

The `lexical_terms()` function converts text into searchable unigrams and
bigrams. A unigram contains one term; a bigram keeps two adjacent terms and
therefore preserves short phrases such as `semantic search`.

Software identifiers are normalized into useful pieces as well. CamelCase and
snake_case names can therefore contribute related lexical terms.


In [ ]:
example = "scanRepository selects source files"

terms = qachat.lexical_terms(example)

print(terms)


In [ ]:
unigrams = [term for term in terms if " " not in term]

print("One-word terms:", unigrams)


In [ ]:
bigrams = [term for term in terms if " " in term]

print("Two-word terms:", bigrams)


## 4. Build the lexical index

The index converts each chunk into a sparse TF-IDF representation. Term
frequency (TF) emphasizes terms that occur within a chunk, while inverse
document frequency (IDF) emphasizes terms that are relatively rare across the
repository. The stored vectors can be reused for later questions.


In [ ]:
lexical_index = qachat.build_lexical_index(chunks)

assert len(lexical_index.chunks) == len(chunks)
assert len(lexical_index.vectors) == len(chunks)

print(f"Indexed chunks: {len(lexical_index.chunks)}")
print(f"Known terms: {len(lexical_index.inverse_document_frequency)}")


In [ ]:
first_vector = lexical_index.vectors[0]
strongest_terms = sorted(
    first_vector.items(),
    key=lambda item: item[1],
    reverse=True,
)

print("Terms from strongest:")
for term in strongest_terms:
    print(term)


## 5. Search by wording

The question remains plain text. It is converted with the same vocabulary and
IDF weights as the indexed chunks, and cosine similarity ranks the chunks that
share the strongest terms and phrases.


In [ ]:
QUESTION = (
    "Which function turns software identifiers into one-word "
    "and two-word terms?"
)

print(QUESTION)


In [ ]:
results = qachat.lexical_search(
    lexical_index,
    QUESTION,
    count=5,
)

results


In [ ]:
scores = [result.score for result in results]

assert results
assert scores == sorted(scores, reverse=True)

print(f"Retrieved chunks: {len(results)}")
print(f"Highest score: {scores[0]}")


## 6. Inspect the lexical ranking

A higher lexical score means stronger overlap in important terms under this
TF-IDF representation. As with semantic similarity, the score orders candidates
inside this channel; it is not a probability of correctness.


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(f"{rank}. {result.score:.3f} | {location} | {name}")


In [ ]:
best = results[0]

print("Most relevant chunk:\n")
print(best.chunk.text)


In [ ]:
for rank, result in enumerate(results):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(f"{result.chunk.text}")


## 7. Search an exact identifier

Exact identifiers are a lexical strength. The query below directly names a
function, so its source chunk should rank strongly without an embedding model.


In [ ]:
identifier_results = qachat.lexical_search(
    lexical_index,
    "build_lexical_index",
    count=3,
)

for rank, result in enumerate(identifier_results):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(result.chunk.text)


## Continue the tutorial series

**Previous:** [Tutorial 70 — Retrieve by semantic similarity](./70_semantic_retrieval.ipynb)

**Next:** [Tutorial 90 — Retrieve through code relationships](./90_code_aware_retrieval.ipynb)
